In [ ]:
import os ##### with thesis-segmentation ENV
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from skimage.transform import AffineTransform, SimilarityTransform, estimate_transform, warp, resize
from skimage.feature import ORB, SIFT, match_descriptors, plot_matched_features
import skimage.color as color
import skimage.measure as measure
import skimage.io as io
from skimage import exposure
from skimage.restoration import denoise_tv_chambolle

from scipy.interpolate import interp1d

from tifffile import imread, imwrite, TiffFile

from __future__ import print_function

import histomicstk as htk

import cv2 as cv


from valis.preprocessing import standardize_colorfulness, norm_img_stats, create_tissue_mask, collect_img_stats
from valis import registration

## Loading Images

In [ ]:
def load_image_data(folder_path):

    img_data = []
    label_data = []
    
    for _, _, files in os.walk(folder_path):
        for index, file in enumerate(files):
            if file.endswith(".tif"): 
                image_path = os.path.join(folder_path, file)
                img_raw = imread(image_path)
                img = np.array(img_raw)[:, :]
                img_data.append(img)

    return img_data

def display_image(img):
    img = img.astype(float)
    img = img / img.max()
    plt.imshow(img, cmap='gray', vmin=0, vmax=1)
    plt.colorbar()                 
    plt.show()


In [ ]:
scale = 0.5023/0.209877

# load dapi
dapi_img_data = load_image_data("../../../Thesis_Data/for_valis/CRC027/cropped/dapi")
dapi1_init = resize(dapi_img_data[0], (int(dapi_img_data[0].shape[0]/scale), int(dapi_img_data[0].shape[1]/scale)), anti_aliasing=True)
dapi2_init = resize(dapi_img_data[1], (int(dapi_img_data[1].shape[0]/scale), int(dapi_img_data[1].shape[1]/scale)), anti_aliasing=True)

# load hne
hne_image_data = load_image_data("../../../Thesis_Data/for_valis/CRC027/cropped/hne")
hne1_init = hne_image_data[0]
hne2_init = hne_image_data[1]


display_image(dapi1_init)
display_image(hne1_init)
display_image(dapi2_init)
display_image(hne2_init)

## Preprocessing

#### Colour Deconvolusion

In [ ]:
def colour_deconvolusion_preprocessing_HnE(hne_init):
    # create stain to color map
    stain_color_map = htk.preprocessing.color_deconvolution.stain_color_map
    print('stain_color_map:', stain_color_map, sep='\n')

    # specify stains of input image
    stains = ['hematoxylin',  # nuclei stain
              'eosin',        # cytoplasm stain
              'null']         # set to null if input contains only two stains

    # create stain matrix
    W = np.array([stain_color_map[st] for st in stains]).T

    # perform standard color deconvolution
    imDeconvolved = htk.preprocessing.color_deconvolution.color_deconvolution(hne_init, W)
    hne_deconv = 1 - imDeconvolved.Stains[:, :, 0]

    return imDeconvolved, hne_deconv


hne1_imDeconvolved, hne1_deconv = colour_deconvolusion_preprocessing_HnE(hne1_init)
# Display results
display_image(hne1_imDeconvolved.Stains[:, :, 0])  # hematoxylin
display_image(hne1_deconv)           
display_image(hne1_imDeconvolved.Stains[:, :, 1])  # eosin

In [ ]:
# for the second cropped image pair

_, hne2_deconv = colour_deconvolusion_preprocessing_HnE(hne2_init)

display_image(hne2_deconv)           

#### VALIS preprocessing

In [ ]:
def VALIS_preprocessing_DAPI(dapi_img):
    _, dapi_img_stats = collect_img_stats(dapi_img)
    dapi_VALIS0 = norm_img_stats(dapi_img*255, target_stats=dapi_img_stats) # expects pixel values 0-255
    dapi_VALIS = denoise_tv_chambolle(dapi_VALIS0, weight=0.01)
    return dapi_VALIS, dapi_img_stats

def VALIS_preprocessing_HnE(hne_img, dapi_img_stats):
    hne_stand = standardize_colorfulness(hne_img, c=0.2, h=0)
    hne_gray_inverted = 1 - color.rgb2gray(hne_stand)
    _, hne_mask = create_tissue_mask(hne_img)
    hne_VALIS0 = norm_img_stats(hne_gray_inverted*255, mask=hne_mask, target_stats=dapi_img_stats)
    hne_VALIS = denoise_tv_chambolle(hne_VALIS0, weight=0.01)
    return hne_VALIS

In [ ]:
dapi1_VALIS, dapi1_img_stats = VALIS_preprocessing_DAPI(dapi1_init)
hne1_VALIS = VALIS_preprocessing_HnE(hne1_init, dapi1_img_stats)

display_image(dapi1_VALIS)
display_image(hne1_VALIS)

## Registration with ORB, RANSAC

In [ ]:
# using ORB and RANSAC for registration

def registration_with_ORB(dapi_img, hne_img, fast_threshold=20):
    dapiX, dapiY = dapi_img.shape
    hneX, hneY = hne_img.shape
    scale_factor = 2  

    # Resize the images
    dapi_scaled = resize(dapi_img, (dapiX / scale_factor, dapiY / scale_factor), anti_aliasing=True)
    hne_scaled = resize(hne_img, (hneX / scale_factor, hneY / scale_factor), anti_aliasing=True)

    # Initialize ORB detector
    orb = ORB(n_keypoints=1000, fast_threshold=fast_threshold)

    # Detect features and descriptors
    orb.detect_and_extract(hne_scaled)
    keypoints1, descriptors1 = orb.keypoints, orb.descriptors

    orb.detect_and_extract(dapi_scaled)
    keypoints2, descriptors2 = orb.keypoints, orb.descriptors

    # Match descriptors
    matches = match_descriptors(descriptors1, descriptors2, cross_check=True)

    # Extract matched keypoints
    src, dst = keypoints1[matches[:, 0]], keypoints2[matches[:, 1]]

    # Plot keypoints and matches
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    plt.gray()
    plot_matched_features(hne_scaled, dapi_scaled, keypoints0=keypoints1, keypoints1=keypoints2, matches=matches, ax=ax)
    ax.axis('off')
    ax.set_title("Keypoint Matches")
    plt.show()

    # Verify number of matches
    print(f"Number of matches: {len(matches)}")

    # Check if we have enough matches to compute a reliable transformation
    if len(matches) < 4:
        raise ValueError("Not enough matches to compute a reliable transformation")

    dst, src = dst * scale_factor, src * scale_factor

    # Compute affine transformation using RANSAC 
    model_robust, inliers = measure.ransac((dst, src),
                                   AffineTransform, min_samples=4,
                                   residual_threshold=2, max_trials=1000)

    return model_robust, inliers

In [ ]:
def overlay_images(dapi_img, hne_img, title='overlay'):
    plt.imshow(dapi_img, cmap='Greens', alpha=1)
    plt.imshow(hne_img, cmap='Reds', alpha=0.4)
    plt.title(title)
    plt.axis('off')
    
    legend_patches = [
        Patch(color='green', label='DAPI', alpha=0.8),
        Patch(color='red', label='HnE', alpha=0.4)
    ]
    plt.legend(handles=legend_patches)


def display_reg_results_with_features(dapi_img, hne_img, model, is_hne_rgb=True):

    registered_image = warp(hne_img, model.inverse, output_shape=dapi_img.shape)
    
    # Display results
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 3, 1)
    plt.title('HnE')
    plt.imshow(hne_img, cmap='gray')

    plt.subplot(1, 3, 2)
    plt.title('DAPI')
    plt.imshow(dapi_img, cmap='gray')

    plt.subplot(1, 3, 3)
    plt.title('Registered Image (aligned version of HnE)')
    plt.imshow(registered_image, cmap='gray')

    plt.show()

    # Display overlay
    plt.figure(figsize=(12,6))
    plt.subplot(1,2,1)
    overlay_images(dapi_img, 1-hne_img[...,0], title='Before Registration Overlay') if is_hne_rgb else overlay_images(dapi_img, hne_img, title='Before Registration Overlay')
    plt.subplot(1,2,2)
    overlay_images(dapi_img, 1-registered_image[...,0], title='After Registration Overlay') if is_hne_rgb else overlay_images(dapi_img, registered_image, title='After Registration Overlay')

    plt.show()

In [ ]:
model, _ = registration_with_ORB(dapi1_VALIS, hne1_VALIS, 20) # with VALIS preprocessing
display_reg_results_with_features(dapi1_init, hne1_init, model)

In [ ]:
model, _ = registration_with_ORB(dapi1_init, hne1_deconv, 0.05) # with colour deconvolution
display_reg_results_with_features(dapi1_init, hne1_init, model)

In [ ]:
dapi_roi_1 = imread("../../../Thesis_Data/for_valis/CRC027/input_images/dapi.tiff") # full images with VALIS preprocessing
hne_roi_1 = imread("../../../Thesis_Data/for_valis/CRC027/input_images/hne.tiff")

model, _ = registration_with_ORB(dapi_roi_1, hne_roi_1, 20)
display_reg_results_with_features(dapi_roi_1, hne_roi_1, model, False)

## Registration with SIFT, RANSAC

In [ ]:
# using SIFT and RANSAC for registration

def registration_with_SIFT(dapi_img, hne_img, max_ratio=0.6, n_octaves=3, n_scales=5):
    dapiX, dapiY = dapi_img.shape
    hneX, hneY = hne_img.shape
    scale_factor = 4

    # Resize the images to reduce memory usage
    dapi_scaled = resize(dapi_img, (dapiX // scale_factor, dapiY // scale_factor), anti_aliasing=True)
    hne_scaled = resize(hne_img, (hneX // scale_factor, hneY // scale_factor), anti_aliasing=True)

    descriptor_extractor = SIFT(n_octaves=n_octaves, n_scales=n_scales)

    descriptor_extractor.detect_and_extract(hne_scaled)
    keypoints1, descriptors1 = descriptor_extractor.keypoints, descriptor_extractor.descriptors

    descriptor_extractor.detect_and_extract(dapi_scaled)
    keypoints2, descriptors2 = descriptor_extractor.keypoints, descriptor_extractor.descriptors

    matches12 = match_descriptors(
        descriptors1, descriptors2, max_ratio=max_ratio, cross_check=True
    )

    # Extract matched keypoints
    src, dst = keypoints1[matches12[:, 0]], keypoints2[matches12[:, 1]]

    fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11, 8))
    plt.gray()
    plot_matched_features(hne_scaled, dapi_scaled, keypoints0=keypoints1, keypoints1=keypoints2, matches=matches12, ax=ax[0])
    ax[0].axis('off')
    ax[0].set_title("H&E vs. DAPI\n" "(all keypoints and matches)")


    plot_matched_features(hne_scaled, dapi_scaled, keypoints0=keypoints1, keypoints1=keypoints2, matches=matches12, ax=ax[1], only_matches=True)
    ax[1].axis('off')
    ax[1].set_title("H&E vs. DAPI\n" "(subset of matches for visibility)")

    plt.tight_layout()
    plt.show()

    dst, src = dst * scale_factor, src * scale_factor

    # Compute affine transformation using RANSAC 
    model_robust, inliers = measure.ransac((dst, src),
                               AffineTransform, min_samples=4,
                               residual_threshold=2, max_trials=1000)
    
    hnetemp_matches = src[inliers] 
    dapitemp_matches = dst[inliers] 
    
    hne_matches = hnetemp_matches[:, [1, 0]].copy()
    dapi_matches = dapitemp_matches[:, [1, 0]].copy()

    return model_robust, [hne_matches, dapi_matches]


In [ ]:
model, _ = registration_with_SIFT(dapi1_VALIS, hne1_VALIS) # with VALIS preprocessing
display_reg_results_with_features(dapi1_init, hne1_init, model)


In [ ]:
hne1_gray = 1 - color.rgb2gray(hne1_init) # H&E with gray scale and inversion
model, _ = registration_with_SIFT(dapi1_init, hne1_gray, 0.8)
display_reg_results_with_features(dapi1_init, hne1_init, model)

In [ ]:
model, [hne1_matches, dapi1_matches] = registration_with_SIFT(dapi1_init, hne1_deconv, 0.6) # with colour deconvolution
display_reg_results_with_features(dapi1_init, hne1_init, model)

In [ ]:
print(model.inverse)

In [ ]:
model, [hne2_matches, dapi2_matches] = registration_with_SIFT(dapi2_init, hne2_deconv, 0.6) # second cropped pair with colour deconvolution
display_reg_results_with_features(dapi2_init, hne2_init, model)

## Registration with ITKElastic

In [ ]:
import importlib
import registration 
importlib.reload(registration)
from registration import registration_with_intensity

In [ ]:
registered_image = warp(hne1_deconv, model.inverse, output_shape=dapi1_init.shape)
img_1_transformations_map, img_1_registered_with_intensity = registration_with_intensity(dapi1_init, hne1_deconv, 0.5023, "../../../Thesis_Data/output", 'rigid')

for img, label in zip(img_1_registered_with_intensity, ['rigid', 'rigid-affine', 'rigid-affine-bspline']):
    overlay_images(dapi1_init, img, label)
    plt.show()


## Initial transformation using computed transformation matrix

#### Using a pair of points

In [ ]:
def initial_transformation_from_features_(dapi_img, hne_img, dapi_matches, hne_matches):
    hne_p1 = np.array(hne_matches[3, :], dtype=np.float32)
    hne_p2 = np.array(hne_matches[1, :], dtype=np.float32)

    dapi_p1 = np.array(dapi_matches[3, :], dtype=np.float32)
    dapi_p2 = np.array(dapi_matches[1, :], dtype=np.float32)

    v_hne = hne_p2 - hne_p1 # hne vector
    v_dapi = dapi_p2 - dapi_p1 # dapi vector

    # If either vector is (near) zero, rotation is undefined
    if np.linalg.norm(v_hne) < 1e-8 or np.linalg.norm(v_dapi) < 1e-8:
        raise ValueError("One of the vectors (top->bottom) has near-zero length.")

    # compute rotation angle (theta)
    angle_hne = np.arctan2(v_hne[1], v_hne[0])
    angle_dapi = np.arctan2(v_dapi[1], v_dapi[0])
    theta = angle_dapi - angle_hne

    c = np.cos(theta)
    s = np.sin(theta)
    R = np.array([[c, -s],
                  [s,  c]])

    # compute translation
    t_total = dapi_p1 - (R @ hne_p1)

    # compute affine matrix
    affine_matrix = np.hstack([R, t_total.reshape(2,1)]).astype(np.float32)
    matrix_skimage = np.vstack([affine_matrix, [0, 0, 1]])

    # transform hne image
    tform = AffineTransform(matrix=matrix_skimage)
    aligned_hne = warp(hne_img, tform.inverse, output_shape=dapi_img.shape)
    # aligned_hne = cv.warpAffine(hne_img, affine_matrix, (dapi_img.shape[1], dapi_img.shape[0]))

    return affine_matrix, aligned_hne

In [ ]:
affine_matrix, hne1_aligned = initial_transformation_from_features_(dapi1_init, hne1_deconv, dapi1_matches, hne1_matches)
overlay_images(dapi1_init, hne1_aligned, title='HnE initial transformation from features')
plt.show()

_, hne2_aligned = initial_transformation_from_features_(dapi2_init, hne2_deconv, dapi2_matches, hne2_matches)
overlay_images(dapi2_init, hne2_aligned, title='HnE initial transformation from features')
plt.show()

#### Using all points

In [ ]:
def initial_transformation_from_features(dapi_img, hne_img, dapi_matches, hne_matches, tform_type='similarity'):

    tform = estimate_transform(tform_type, src=hne_matches, dst=dapi_matches)
    aligned_hne = warp(hne_img, tform.inverse, output_shape=dapi_img.shape)

    return tform.params[:2, :], aligned_hne

affine_matrix, hne1_aligned = initial_transformation_from_features(dapi1_init, hne1_deconv, dapi1_matches, hne1_matches)
overlay_images(dapi1_init, hne1_aligned, title='HnE initial transformation from features')
plt.show()

affine_matrix, hne2_aligned = initial_transformation_from_features(dapi2_init, hne2_deconv, dapi2_matches, hne2_matches)
overlay_images(dapi2_init, hne2_aligned, title='HnE initial transformation from features')
plt.show()

## Registration with VALIS

In [ ]:
imwrite("../../../Thesis_Data/for_valis/CRC027/cropped_for_valis/hne.tiff", hne1_deconv)
imwrite("../../../Thesis_Data/for_valis/CRC027/cropped_for_valis/dapi.tiff", dapi1_init*255)

In [ ]:
# Registration with VALIS
slide_src_dir = "../../../Thesis_Data/for_valis/CRC027/cropped_for_valis"
results_dst_dir = "../../../Thesis_Data/for_valis/CRC027/results_cropped"
registered_slide_dst_dir = "../../../Thesis_Data/for_valis/CRC027/registered_cropped"


reg = registration.serial_rigid.register_images(
    slide_src_dir,
    results_dst_dir,  
)

non_rigid_registrar = registration.serial_non_rigid.register_images(reg)

In [ ]:
from valis import non_rigid_registrars

registrar = non_rigid_registrars.NonRigidRegistrar()
dapi1_resized = resize(dapi1_VALIS, hne1_VALIS.shape, anti_aliasing=True)
_, hne_mask = create_tissue_mask(hne1_init)
warped_img, warped_grid, bk_dxdy = registrar.register(dapi1_resized, hne1_VALIS, mask=hne_mask)
display_image(warped_img)